# 02 — Temporal Feature Engineering

Computes eight history-dependent edge features using a single chronological streaming pass over all transactions. Because every feature for a transaction at time `t` is built exclusively from events strictly earlier than `t`, there is no look-ahead leakage.

**Input:** `labeled_transactions_april_sept.csv`  
**Output:** `feature_engineered_transactions.csv`

---

### Features computed (8 history features + 1 value feature = 9 total)

| Feature | Description |
|---------|-------------|
| `log_src_tx_count_past` | Log of sender's prior transaction count |
| `log_dst_tx_count_past` | Log of receiver's prior transaction count |
| `log_src_to_dst_count_past` | Log of prior directed interactions from sender to receiver |
| `log_dst_to_src_count_past` | Log of prior directed interactions from receiver to sender |
| `log_time_since_src_last` | Log of seconds since sender's last transaction |
| `log_time_since_dst_last` | Log of seconds since receiver's last transaction |
| `log_src_value_sum_past` | Log of sender's cumulative transaction value |
| `log_dst_value_sum_past` | Log of receiver's cumulative transaction value |
| `log_transaction_value` | Log of the current transaction value |

All raw counts and values are heavy-tailed, so each is passed through `log(1+x)` before use.

## 1. Load labeled transactions

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

print("Loading labeled transactions...")

df = pd.read_csv("labeled_transactions_april_sept.csv")

# Ensure chronological order — critical for leakage-free feature engineering
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded: {df.shape[0]:,} transactions, {df.shape[1]} columns")
print(f"Wash trading: {df['is_wash_trading'].sum():,} ({df['is_wash_trading'].mean():.3%})")

## 2. Streaming feature computation

Single pass through all transactions in chronological order. For each transaction, features are read from the current historical state **before** updating it, ensuring strict look-ahead prevention.

In [ ]:
n = len(df)

# Output arrays
src_tx_count_past     = np.zeros(n, dtype=np.int32)
dst_tx_count_past     = np.zeros(n, dtype=np.int32)
src_to_dst_count_past = np.zeros(n, dtype=np.int32)
dst_to_src_count_past = np.zeros(n, dtype=np.int32)
time_since_src_last   = np.zeros(n, dtype=np.float32)
time_since_dst_last   = np.zeros(n, dtype=np.float32)
src_value_sum_past    = np.zeros(n, dtype=np.float32)
dst_value_sum_past    = np.zeros(n, dtype=np.float32)

# Historical state trackers
wallet_tx_count  = defaultdict(int)
wallet_value_sum = defaultdict(float)
wallet_last_time = {}
pair_count       = defaultdict(int)

print(f"Streaming through {n:,} transactions...")

for i, row in enumerate(df.itertuples(index=False)):
    src = row.from_address
    dst = row.to_address
    ts  = row.timestamp
    val = row.transaction_value

    # Read historical state (before this transaction)
    src_tx_count_past[i]     = wallet_tx_count[src]
    dst_tx_count_past[i]     = wallet_tx_count[dst]
    src_value_sum_past[i]    = wallet_value_sum[src]
    dst_value_sum_past[i]    = wallet_value_sum[dst]
    src_to_dst_count_past[i] = pair_count[(src, dst)]
    dst_to_src_count_past[i] = pair_count[(dst, src)]

    if src in wallet_last_time:
        time_since_src_last[i] = ts - wallet_last_time[src]
    if dst in wallet_last_time:
        time_since_dst_last[i] = ts - wallet_last_time[dst]

    # Update historical state (after reading)
    wallet_tx_count[src]  += 1
    wallet_tx_count[dst]  += 1
    wallet_value_sum[src] += val
    wallet_value_sum[dst] += val
    wallet_last_time[src]  = ts
    wallet_last_time[dst]  = ts
    pair_count[(src, dst)] += 1

    if i % 500000 == 0 and i > 0:
        print(f"  {i:,} / {n:,}")

print("Done.")

## 3. Attach features and apply log transform

In [ ]:
# Attach raw features
df["src_tx_count_past"]     = src_tx_count_past
df["dst_tx_count_past"]     = dst_tx_count_past
df["src_to_dst_count_past"] = src_to_dst_count_past
df["dst_to_src_count_past"] = dst_to_src_count_past
df["time_since_src_last"]   = time_since_src_last
df["time_since_dst_last"]   = time_since_dst_last
df["src_value_sum_past"]    = src_value_sum_past
df["dst_value_sum_past"]    = dst_value_sum_past

# Apply log(1+x) transform to all 8 history features
history_features = [
    "src_tx_count_past",
    "dst_tx_count_past",
    "src_to_dst_count_past",
    "dst_to_src_count_past",
    "time_since_src_last",
    "time_since_dst_last",
    "src_value_sum_past",
    "dst_value_sum_past"
]

for col in history_features:
    df[f"log_{col}"] = np.log1p(df[col])

# 9th feature: log transaction value
df["log_transaction_value"] = np.log1p(df["transaction_value"])

final_features = [f"log_{c}" for c in history_features] + ["log_transaction_value"]
print(f"Final edge features ({len(final_features)}): {final_features}")

## 4. Save output

In [ ]:
df.to_csv("feature_engineered_transactions.csv", index=False)

print(f"Saved: feature_engineered_transactions.csv")
print(f"Shape: {df.shape}")
print("\nNext step: run 03_graph_dataset_building.ipynb")